# Mizan F8: train the local verifier (Qwen3.5-2B + LoRA)

Runs on a free **Colab T4** or **Kaggle T4** GPU (16 GB). It takes about 1–1.5 hours.

1. **Runtime → Change runtime type → T4 GPU** (Colab). On Kaggle: **Settings → Accelerator → GPU T4**.
2. Run the cells in order. Cell 3 asks for `bundle.tgz`:
   - on Colab, upload it when prompted;
   - on Kaggle, add it as a dataset first.
3. The last cell produces `verifier_results.tgz`. Download it and give it back.

The bundle has the code and the training data. Nothing leaves the notebook except the results
file you download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Where am I, and where is the bundle?
import glob, os, tarfile

ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
os.chdir(WORK)
if ON_KAGGLE:
    found = glob.glob("/kaggle/input/**/bundle.tgz", recursive=True)
    assert found, "Add bundle.tgz as a Kaggle dataset (Add Input -> Upload)"
    bundle = found[0]
else:
    from google.colab import files
    uploaded = files.upload()  # choose bundle.tgz
    bundle = next(iter(uploaded))
with tarfile.open(bundle) as tar:
    tar.extractall(WORK)
os.chdir(os.path.join(WORK, "bundle"))
print(os.getcwd(), sorted(os.listdir(".")))

In [ ]:
# Colab ships an old torchao that peft refuses to import.
!pip uninstall -y -q torchao
!pip install -q "transformers>=5" "peft>=0.15" structlog opentelemetry-api pydantic accelerate
import torch, transformers, peft
print(torch.__version__, torch.cuda.is_available(), transformers.__version__, peft.__version__)

In [ ]:
# Train (3 epochs, checkpoint chosen on validation), score the test split once,
# then measure throughput.
!python pod_verifier.py 2>&1 | grep -E --line-buffered 'verifier_|"train"|best_epoch|throughput|base_failed|Error|Traceback'

In [ ]:
# Pack the results (the adapter, about 70 MB, is included for serving).
import tarfile
with tarfile.open("verifier_results.tgz", "w:gz") as tar:
    for name in ["runs/verifier/report.json", "runs/verifier/adapter",
                 "verifier_test.jsonl", "verifier_throughput.json"]:
        if os.path.exists(name):
            tar.add(name)
print(os.path.getsize("verifier_results.tgz") // 1024, "KB")
if not ON_KAGGLE:
    from google.colab import files
    files.download("verifier_results.tgz")
else:
    import shutil
    shutil.copy("verifier_results.tgz", "/kaggle/working/verifier_results.tgz")
    print("Download it from the Output tab: /kaggle/working/verifier_results.tgz")